In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path



INPUT_FILE = Path(r"D:\BORUE\BRUSE_Brave_Evaluated_200.xlsx")

OUTPUT_DIR = INPUT_FILE.resolve().parent



df = pd.read_excel(
    INPUT_FILE,
    sheet_name="Per_Query_Metrics"
)

conditions = [
    "common",
    "bruse_query2",
    "bruse_query4",
    "bruse_query6",
    "bruse_query_full"
]

metrics = [
    "P@10",
    "MRR@10",
    "nDCG@10",
    "Success@10"
]


df = df[df["condition"].isin(conditions)].copy()


counts = df.groupby("condition").size().reindex(conditions)

print("Queries per condition:")
print(counts.to_string())

assert (counts == 201).all(), (
    "Expected exactly 201 queries per condition."
)


means = (
    df.groupby("condition")[metrics]
    .mean()
    .reindex(conditions)
)

print("\nMean retrieval scores:")
print(means.round(4).to_string())


colors = [
    "#D9ECFC",
    "#B5DEFA",
    "#8BC8F2",
    "#69B3E7",
    "#4A9BD8"
]

x = np.arange(len(metrics))
bar_width = 0.16

fig, ax = plt.subplots(
    figsize=(13, 6.5),
    dpi=150
)



for i, condition in enumerate(conditions):

    values = means.loc[condition, metrics].values

    positions = x + (i - 2) * bar_width

    bars = ax.bar(
        positions,
        values,
        width=bar_width,
        color=colors[i],
        label=condition
    )


    for bar, value in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.008,
            f"{value:.3f}",
            ha="center",
            va="bottom",
            fontsize=9,
            color="black"
        )



ax.set_ylabel("Mean Score", fontsize=12)

ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=11)

ax.set_ylim(0, 1.08)
ax.set_yticks(np.arange(0, 1.01, 0.2))

ax.tick_params(axis="y", labelsize=10)

ax.grid(
    axis="y",
    linestyle="--",
    linewidth=0.7,
    alpha=0.25
)

ax.set_axisbelow(True)

ax.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.12),
    ncol=5,
    frameon=False,
    fontsize=10
)

# Thin border around plot
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(0.8)

plt.tight_layout()



png_file = OUTPUT_DIR / "BRUSE_Pilot_201_Results.png"
pdf_file = OUTPUT_DIR / "BRUSE_Pilot_201_Results.pdf"

fig.savefig(
    png_file,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

fig.savefig(
    pdf_file,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

print("\nGraph saved successfully:")
print(png_file)
print(pdf_file)
